# WWW 2025 EReL@MIR — MM-CTR Challenge
## Task 2: Multimodal CTR Prediction — v3 (Memory Efficient)

### Key design decisions for memory efficiency
- **No pandas merge of embeddings** — embeddings stay as a PyTorch lookup matrix indexed by item_id. The 3.6M row train DataFrame never gets 128 extra columns added to it.
- **No item_seq / DIN** — too large to load on 16GB RAM. Skipped entirely.
- **Training on 30% sample** — reduces memory and time while still giving good AUC. Full test set is always predicted.
- **Embedding lookup happens inside the Dataset** — only the item indices travel through pandas, the actual vectors are fetched by PyTorch at batch time.

## 1. Imports

In [1]:
import os
import gc
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Limit pyarrow threads to reduce memory spikes on load
import pyarrow as pa
pa.set_cpu_count(1)
pa.set_io_thread_count(1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

PyTorch : 2.11.0+cpu
Device  : cpu


## 2. Configuration

In [2]:
USE_SAMPLE  = True
SAMPLE_FRAC = 0.8
RANDOM_SEED = 42

ID_EMB_DIM   = 48
MM_EMB_DIM   = 128
NUM_EXTRA_FEATURES = 2   # likes_level + views_level
MLP_LAYERS   = [384, 192, 96]
DROPOUT      = 0.5

LEARNING_RATE  = 2e-4
WEIGHT_DECAY   = 3e-4
BATCH_SIZE     = 2048
NUM_EPOCHS     = 35
PATIENCE       = 7

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Config ready.")

Config ready.


## 3. Set Working Directory

In [3]:
os.chdir(r"C:\Users\User\Desktop\MM_CTR_Task2")
print("Directory :", os.getcwd())
print("Files     :", os.listdir())

Directory : C:\Users\User\Desktop\MM_CTR_Task2
Files     : ['.ipynb_checkpoints', 'item_emb.parquet', 'item_info.parquet', 'item_seq.parquet', 'mmctr_env', 'MM_CTR_Task2_Notebook.ipynb', 'MM_CTR_Task2_v2.ipynb', 'MM_CTR_Task2_v3.ipynb', 'MM_CTR_Task2_v4_CPU_HistoryLite.ipynb', 'MM_CTR_Task2_v5_CPU_MemorySafe_StrongEnsemble.ipynb', 'MM_CTR_Task2_v6_SimpleStrong_CPU_Safe.ipynb', 'prediction.csv', 'prediction.zip', 'Presentation_Script_Rehearsal.md', 'test.parquet', 'train.parquet', 'valid.parquet']


## 4. Load Data

We load only lightweight files into pandas. The item embeddings are handled separately and never merged into the main DataFrames.

In [4]:
def reduce_mem(df):
    """Downcast numeric columns to save ~40% RAM."""
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df

print("Loading click data...")
t0 = time.time()

train = reduce_mem(pd.read_parquet("train.parquet"))
gc.collect()
print(f"train : {train.shape}")

valid = reduce_mem(pd.read_parquet("valid.parquet"))
gc.collect()
print(f"valid : {valid.shape}")

test  = reduce_mem(pd.read_parquet("test.parquet"))
gc.collect()
print(f"test  : {test.shape}")

print(f"\nLoaded in {time.time()-t0:.1f}s")
print("\ntrain columns:", train.columns.tolist())
print(train.head(3))

Loading click data...
train : (3600000, 6)
valid : (10000, 6)
test  : (379142, 6)

Loaded in 24.3s

train columns: ['user_id', 'item_seq', 'item_id', 'likes_level', 'views_level', 'label']
   user_id                                           item_seq  item_id  \
0   861687  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...    33811   
1   210622  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...    12875   
2   861880  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...     1678   

   likes_level  views_level  label  
0            4            4      0  
1            9            5      1  
2            9            8      1  


## 5. Load Item Embeddings into a Lookup Matrix

Instead of merging 128 embedding columns into the 3.6M row train DataFrame (which would require ~1.8GB extra RAM), we:
1. Load item_emb in batches
2. Build a NumPy matrix of shape `(num_items, 128)`
3. Convert it to a PyTorch tensor
4. Look up embeddings by item index inside the Dataset — zero pandas overhead

In [5]:
print("Loading item embeddings in batches...")
t0 = time.time()

pf = pq.ParquetFile("item_emb.parquet")
batches = []
for batch in pf.iter_batches(batch_size=10000):
    batches.append(batch.to_pandas())
    gc.collect()

item_emb = pd.concat(batches, ignore_index=True)
del batches
gc.collect()

print(f"item_emb shape   : {item_emb.shape}")
print(f"item_emb columns : {item_emb.columns.tolist()}")
print(f"Loaded in {time.time()-t0:.1f}s")

Loading item embeddings in batches...
item_emb shape   : (91717, 5)
item_emb columns : ['item_id', 'item_emb_d128_v1', 'item_emb_d128_v2', 'item_emb_d128_v3', 'item_emb_d128_e4']
Loaded in 4.4s


## 6. Detect Columns & Encode IDs

In [6]:
def detect_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

USER_COL  = detect_col(train, ['user_id','user','uid','userId'])
ITEM_COL  = detect_col(train, ['item_id','item','iid','itemId'])
LABEL_COL = detect_col(train, ['label','click','target','y'])
EMB_ID_COL = detect_col(item_emb, ['item_id','item','iid','itemId'])

# ── MANUAL OVERRIDE if needed ──────────────────────────────────────────────
# USER_COL   = 'user_id'
# ITEM_COL   = 'item_id'
# LABEL_COL  = 'label'
# EMB_ID_COL = 'item_id'

print(f"USER_COL   : {USER_COL}")
print(f"ITEM_COL   : {ITEM_COL}")
print(f"LABEL_COL  : {LABEL_COL}")
print(f"EMB_ID_COL : {EMB_ID_COL}")
assert all(v is not None for v in [USER_COL, ITEM_COL, LABEL_COL, EMB_ID_COL])

# ── Encode user and item IDs ───────────────────────────────────────────────
all_users = pd.concat([train[USER_COL], valid[USER_COL], test[USER_COL]]).unique()
all_items = pd.concat([train[ITEM_COL], valid[ITEM_COL], test[ITEM_COL]]).unique()

user_le = LabelEncoder().fit(all_users)
item_le = LabelEncoder().fit(all_items)

NUM_USERS = len(user_le.classes_)
NUM_ITEMS = len(item_le.classes_)

for df in [train, valid, test]:
    df['user_idx'] = user_le.transform(df[USER_COL]).astype(np.int32)
    df['item_idx'] = item_le.transform(df[ITEM_COL]).astype(np.int32)

gc.collect()
print(f"\nNUM_USERS : {NUM_USERS}")
print(f"NUM_ITEMS : {NUM_ITEMS}")
print("ID encoding done.")

USER_COL   : user_id
ITEM_COL   : item_id
LABEL_COL  : label
EMB_ID_COL : item_id

NUM_USERS : 1000000
NUM_ITEMS : 82572
ID encoding done.


## 7. Build Embedding Lookup Matrix

Build a single matrix `EMB_MATRIX` of shape `(NUM_ITEMS+1, 128)` where row `i` is the multimodal embedding of item with index `i`. Index 0 is reserved as padding (zero vector) for unknown items.

In [7]:
# Detect which column holds the embedding arrays
EMB_VEC_COL = [c for c in item_emb.columns if c != EMB_ID_COL][0]
print(f"Using embedding column: {EMB_VEC_COL}")

# Use e4 if available (likely the best combined embedding), else first available
emb_cols_available = [c for c in item_emb.columns if c != EMB_ID_COL]
preferred = [c for c in emb_cols_available if 'e4' in c]
EMB_VEC_COL = preferred[0] if preferred else emb_cols_available[0]
print(f"Selected embedding column: {EMB_VEC_COL}")

# Build the lookup matrix
EMB_MATRIX = np.zeros((NUM_ITEMS + 1, MM_EMB_DIM), dtype=np.float32)

known_items = set(item_le.classes_)
for _, row in item_emb.iterrows():
    iid = row[EMB_ID_COL]
    if iid in known_items:
        idx = item_le.transform([iid])[0]
        vec = np.array(row[EMB_VEC_COL], dtype=np.float32)
        # L2 normalize
        norm = np.linalg.norm(vec)
        if norm > 1e-8:
            vec = vec / norm
        EMB_MATRIX[idx] = vec

# Free item_emb — no longer needed
del item_emb
gc.collect()

print(f"EMB_MATRIX shape : {EMB_MATRIX.shape}")
print(f"Non-zero rows    : {(EMB_MATRIX.sum(axis=1) != 0).sum()}")
print("Embedding matrix ready.")

Using embedding column: item_emb_d128_v1
Selected embedding column: item_emb_d128_e4
EMB_MATRIX shape : (82573, 128)
Non-zero rows    : 82572
Embedding matrix ready.


## 8. Sample Training Data & Prepare Arrays

In [8]:
if USE_SAMPLE:
    train_df = train.sample(frac=SAMPLE_FRAC, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"[SAMPLE MODE] Training on {len(train_df):,} / {len(train):,} rows ({SAMPLE_FRAC*100:.0f}%)")
else:
    train_df = train.copy()
    print(f"Full training set: {len(train_df):,} rows")

EXTRA_COLS = ['likes_level', 'views_level']

for c in EXTRA_COLS:
    assert c in train.columns, f"Missing column: {c}"

mean_vals = train_df[EXTRA_COLS].mean()
std_vals  = train_df[EXTRA_COLS].std().replace(0, 1)

tr_user  = train_df['user_idx'].values.astype(np.int32)
tr_item  = train_df['item_idx'].values.astype(np.int32)
tr_extra = ((train_df[EXTRA_COLS] - mean_vals) / std_vals).values.astype(np.float32)
tr_label = train_df[LABEL_COL].values.astype(np.float32)

va_user  = valid['user_idx'].values.astype(np.int32)
va_item  = valid['item_idx'].values.astype(np.int32)
va_extra = ((valid[EXTRA_COLS] - mean_vals) / std_vals).values.astype(np.float32)
va_label = valid[LABEL_COL].values.astype(np.float32)

te_user  = test['user_idx'].values.astype(np.int32)
te_item  = test['item_idx'].values.astype(np.int32)
te_extra = ((test[EXTRA_COLS] - mean_vals) / std_vals).values.astype(np.float32)

del train_df
gc.collect()

print(f"\ntrain arrays : {tr_user.shape}")
print(f"valid arrays : {va_user.shape}")
print(f"test  arrays : {te_user.shape}")
print(f"extra features: {EXTRA_COLS}")
print(pd.Series(tr_label).value_counts())

[SAMPLE MODE] Training on 2,880,000 / 3,600,000 rows (80%)

train arrays : (2880000,)
valid arrays : (10000,)
test  arrays : (379142,)
extra features: ['likes_level', 'views_level']
0.0    2160108
1.0     719892
Name: count, dtype: int64


## 9. PyTorch Dataset & DataLoader

The Dataset looks up the multimodal embedding from `EMB_MATRIX` at batch time using the item index. This means the large embedding matrix lives only in NumPy/PyTorch memory — not duplicated across the pandas DataFrame.

In [9]:
EMB_TENSOR = torch.FloatTensor(EMB_MATRIX)
del EMB_MATRIX
gc.collect()
print(f"EMB_TENSOR shape: {EMB_TENSOR.shape}")


class CTRDataset(Dataset):
    def __init__(self, user_arr, item_arr, extra_arr, emb_tensor, label_arr=None):
        self.user  = torch.from_numpy(user_arr).long()
        self.item  = torch.from_numpy(item_arr).long()
        self.extra = torch.from_numpy(extra_arr).float()
        self.emb_t = emb_tensor
        self.label = torch.from_numpy(label_arr).float() if label_arr is not None else torch.zeros(len(user_arr))

    def __len__(self):
        return len(self.user)

    def __getitem__(self, idx):
        u = self.user[idx]
        i = self.item[idx]
        x = self.extra[idx]
        e = self.emb_t[i]
        l = self.label[idx]
        return u, i, x, e, l


train_ds = CTRDataset(tr_user, tr_item, tr_extra, EMB_TENSOR, tr_label)
valid_ds = CTRDataset(va_user, va_item, va_extra, EMB_TENSOR, va_label)
test_ds  = CTRDataset(te_user, te_item, te_extra, EMB_TENSOR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Valid batches : {len(valid_loader)}")
print(f"Test  batches : {len(test_loader)}")

EMB_TENSOR shape: torch.Size([82573, 128])
Train batches : 1407
Valid batches : 5
Test  batches : 186


## 10. Model — Residual MLP with Feature Interactions

Without user history we focus on making the model itself stronger:
- **Feature interaction layer** — explicit cross between user embedding and item multimodal embedding
- **Residual MLP** — skip connections prevent vanishing gradients in deeper networks
- **Multiple interaction terms** — inner product, element-wise product, and concatenation all fed to MLP

In [10]:
class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.fc1  = nn.Linear(in_dim, out_dim)
        self.bn1  = nn.BatchNorm1d(out_dim)
        self.fc2  = nn.Linear(out_dim, out_dim)
        self.bn2  = nn.BatchNorm1d(out_dim)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.drop(out)
        out = self.bn2(self.fc2(out))
        return F.relu(out + residual)


class MultimodalCTR(nn.Module):
    def __init__(self, num_users, num_items, id_emb_dim, mm_emb_dim, extra_dim, mlp_layers, dropout):
        super().__init__()

        self.user_emb = nn.Embedding(num_users + 1, id_emb_dim)
        self.item_emb = nn.Embedding(num_items + 1, id_emb_dim)

        self.mm_proj = nn.Sequential(
            nn.Linear(mm_emb_dim, id_emb_dim),
            nn.LayerNorm(id_emb_dim),
            nn.ReLU()
        )

        input_dim = id_emb_dim * 5 + 1 + extra_dim

        blocks = []
        in_d = input_dim
        for out_d in mlp_layers:
            blocks.append(ResidualBlock(in_d, out_d, dropout))
            in_d = out_d

        self.mlp = nn.Sequential(*blocks)
        self.output = nn.Linear(in_d, 1)

    def forward(self, user_idx, item_idx, extra, mm_emb):
        u  = self.user_emb(user_idx)
        i  = self.item_emb(item_idx)
        mm = self.mm_proj(mm_emb)

        ui_prod  = u * i
        umm_prod = u * mm
        dot = (u * mm).sum(dim=1, keepdim=True)

        x = torch.cat([u, i, mm, ui_prod, umm_prod, dot, extra], dim=1)
        x = self.mlp(x)
        return torch.sigmoid(self.output(x).squeeze(-1))


model = MultimodalCTR(
    num_users  = NUM_USERS,
    num_items  = NUM_ITEMS,
    id_emb_dim = ID_EMB_DIM,
    mm_emb_dim = MM_EMB_DIM,
    extra_dim  = NUM_EXTRA_FEATURES,
    mlp_layers = MLP_LAYERS,
    dropout    = DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
print(model)

Trainable parameters: 52,539,121
MultimodalCTR(
  (user_emb): Embedding(1000001, 48)
  (item_emb): Embedding(82573, 48)
  (mm_proj): Sequential(
    (0): Linear(in_features=128, out_features=48, bias=True)
    (1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
  )
  (mlp): Sequential(
    (0): ResidualBlock(
      (fc1): Linear(in_features=243, out_features=384, bias=True)
      (bn1): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=384, out_features=384, bias=True)
      (bn2): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop): Dropout(p=0.5, inplace=False)
      (skip): Linear(in_features=243, out_features=384, bias=True)
    )
    (1): ResidualBlock(
      (fc1): Linear(in_features=384, out_features=192, bias=True)
      (bn1): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=192, out_featu

## 11. Train with Early Stopping

In [11]:
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE,
                             weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-5
)


def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for user, item, extra, emb, label in loader:
        user  = user.to(device)
        item  = item.to(device)
        extra = extra.to(device)
        emb   = emb.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        preds = model(user, item, extra, emb)

        loss = F.binary_cross_entropy(preds, label)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(label)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for user, item, extra, emb, label in loader:
            user  = user.to(device)
            item  = item.to(device)
            extra = extra.to(device)
            emb   = emb.to(device)

            p = model(user, item, extra, emb).cpu().numpy()
            preds_all.extend(p)
            labels_all.extend(label.numpy())

    return np.array(preds_all), np.array(labels_all)

best_auc         = 0.0
best_state       = None
patience_counter = 0

print(f"{'Epoch':>5} {'Train Loss':>12} {'Val AUC':>10} {'Time':>8}")
print("-" * 42)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    loss = train_epoch(model, train_loader, optimizer, DEVICE)
    vp, vl = evaluate(model, valid_loader, DEVICE)
    auc = roc_auc_score(vl, vp)
    elapsed = time.time() - t0

    flag = ""
    if auc > best_auc:
        best_auc   = auc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        flag = " ← best"
    else:
        patience_counter += 1
        flag = f" (no improve {patience_counter}/{PATIENCE})"

    print(f"{epoch:>5} {loss:>12.5f} {auc:>10.5f} {elapsed:>7.1f}s{flag}")
    scheduler.step()

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest Validation AUC: {best_auc:.5f}")

Epoch   Train Loss    Val AUC     Time
------------------------------------------
    1      0.54609    0.54800  1097.6s ← best
    2      0.48772    0.60144  1073.7s ← best
    3      0.44367    0.60951   764.6s ← best
    4      0.39891    0.61660   673.3s ← best
    5      0.35810    0.62528   677.1s ← best
    6      0.32765    0.64025   681.5s ← best
    7      0.30316    0.65152   688.4s ← best
    8      0.28078    0.66305   680.2s ← best
    9      0.25892    0.66812   706.6s ← best
   10      0.23416    0.66956   743.7s ← best
   11      0.20650    0.67769   789.7s ← best
   12      0.17626    0.68726   805.6s ← best
   13      0.15001    0.70477   811.4s ← best
   14      0.12782    0.73034   975.7s ← best
   15      0.10947    0.75089  1078.9s ← best
   16      0.09426    0.75898  1008.1s ← best
   17      0.08261    0.76838  1012.7s ← best
   18      0.07325    0.78326  1020.1s ← best
   19      0.06600    0.77002  1025.6s (no improve 1/7)
   20      0.06016    0.77763  103

## 12. Final Validation AUC

In [12]:
if best_state:
    model.load_state_dict(best_state)
    print("Best checkpoint loaded.")

vp_final, vl_final = evaluate(model, valid_loader, DEVICE)
final_auc = roc_auc_score(vl_final, vp_final)

print(f"\n{'='*45}")
print(f"  Final Validation AUC : {final_auc:.5f}")
print(f"{'='*45}")
print(f"  Min  : {vp_final.min():.4f}")
print(f"  Max  : {vp_final.max():.4f}")
print(f"  Mean : {vp_final.mean():.4f}")

Best checkpoint loaded.

  Final Validation AUC : 0.78326
  Min  : 0.0001
  Max  : 0.9998
  Mean : 0.5949


## 13. Full Test Set Prediction

Always predicts the entire test set regardless of USE_SAMPLE setting.

In [13]:
model.eval()
test_preds = []

with torch.no_grad():
    for user, item, extra, emb, _ in test_loader:
        user  = user.to(DEVICE)
        item  = item.to(DEVICE)
        extra = extra.to(DEVICE)
        emb   = emb.to(DEVICE)

        p = model(user, item, extra, emb).cpu().numpy()
        test_preds.extend(p)   # IMPORTANT LINE

test_preds = np.array(test_preds)

assert len(test_preds) == len(test), f"Mismatch! preds={len(test_preds)}, test={len(test)}"
assert test_preds.min() >= 0.0 and test_preds.max() <= 1.0

print(f"Test predictions : {len(test_preds):,}")
print(f"Value range      : [{test_preds.min():.4f}, {test_preds.max():.4f}]")
print("All checks passed.")

Test predictions : 379,142
Value range      : [0.0000, 0.9998]
All checks passed.


## 14. Create Submission File

In [14]:
# Task1 and Task1&2 filled with Task2 predictions as placeholders
# This notebook implements Task 2 only
submission = pd.DataFrame({
    "ID"      : np.arange(len(test_preds)),
    "Task1"   : test_preds,
    "Task2"   : test_preds,
    "Task1&2" : test_preds
})

submission.to_csv("prediction.csv", index=False)

with zipfile.ZipFile("prediction.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.csv")

print("prediction.csv saved.")
print("prediction.zip saved.")
print(submission.head())

prediction.csv saved.
prediction.zip saved.
   ID     Task1     Task2   Task1&2
0   0  0.995052  0.995052  0.995052
1   1  0.997469  0.997469  0.997469
2   2  0.902136  0.902136  0.902136
3   3  0.001312  0.001312  0.001312
4   4  0.003693  0.003693  0.003693


## 15. Final Summary

In [15]:
sub = pd.read_csv("prediction.csv")
print("=" * 50)
print("FINAL SUBMISSION SUMMARY")
print("=" * 50)
print(f"Submission rows       : {len(sub):,}")
print(f"Test set rows         : {len(test):,}")
print(f"Match                 : {len(sub) == len(test)}")
print(f"Columns               : {sub.columns.tolist()}")
print(f"Final Validation AUC  : {final_auc:.5f}")
print(f"Pred min / max        : {sub['Task2'].min():.4f} / {sub['Task2'].max():.4f}")
print(f"prediction.csv exists : {os.path.exists('prediction.csv')}")
print(f"prediction.zip exists : {os.path.exists('prediction.zip')}")
print("=" * 50)
print(sub.head())

FINAL SUBMISSION SUMMARY
Submission rows       : 379,142
Test set rows         : 379,142
Match                 : True
Columns               : ['ID', 'Task1', 'Task2', 'Task1&2']
Final Validation AUC  : 0.78326
Pred min / max        : 0.0000 / 0.9998
prediction.csv exists : True
prediction.zip exists : True
   ID     Task1     Task2   Task1&2
0   0  0.995052  0.995052  0.995052
1   1  0.997469  0.997469  0.997469
2   2  0.902136  0.902136  0.902136
3   3  0.001312  0.001312  0.001312
4   4  0.003693  0.003693  0.003693
